# HDI Cross-Country Analysis (2022): A Multi-Source Data Pipeline & Structural Audit
## Notebook 02: Data cleaning

**Goal:** Clean, reshape, and merge all raw datasets into one master dataset ready for analysis.

**Input files:**
- `raw_data/worldbank_raw.csv`
- `raw_data/undp_hdi_raw.csv`
- `raw_data/wgi_government_effectiveness_raw.zip`

**Output file:**
- `processed_data/master_dataset.csv`

## Cleaning Strategy

**Anchor dataset:**
UNDP provides the most stable and consistently reported country-level indicators in this project. GDP per capita is used as strategic proxy to avoid circular bias, since HDI already incorporates GNI per capita.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path                            # file system navigation across any operating system

BASE_DIR = Path("..").resolve()                     # go one folder up from /notebooks to /hdi_project
RAW_DATA = BASE_DIR / "raw_data"                    # input files 
PROCESSED_DATA = BASE_DIR / "processed_data"        # cleaned output files
PROCESSED_DATA.mkdir(exist_ok=True)                 # create folder if it doesn't exist

print("Libraries loaded")
print(f"Raw data folder: {RAW_DATA}")
print(f"Processed data folder: {PROCESSED_DATA}")

Libraries loaded
Raw data folder: E:\Portfolio\projects\HDI_Analysis\raw_data
Processed data folder: E:\Portfolio\projects\HDI_Analysis\processed_data


## I Clean World Bank Data
Source: `worldbank_raw.csv`
Indicators: GDP per capita, Education Expenditure, Health Expenditure

In [2]:
# Load World Bank raw data 
wb_raw = pd.read_csv(RAW_DATA / "worldbank_raw.csv")

wb_raw.shape
wb_raw.head()

,economy,series,Country,Series,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024,YR2025
0,ZWE,NY.GDP.PCAP.KD,Zimbabwe,GDP per capita (constant 2015 US$),1357.530878,1230.819334,1312.200014,1369.197615,1418.471846,1417.718941,NaN
1,ZMB,NY.GDP.PCAP.KD,Zambia,GDP per capita (constant 2015 US$),1301.181330,1228.735001,1269.108218,1298.848050,1330.859982,1343.389664,NaN
2,YEM,NY.GDP.PCAP.KD,"Yemen, Rep.",GDP per capita (constant 2015 US$),NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PSE,NY.GDP.PCAP.KD,West Bank and Gaza,GDP per capita (constant 2015 US$),3378.434621,2922.468011,3051.486070,3099.960901,2888.763061,2072.090195,NaN
4,VIR,NY.GDP.PCAP.KD,Virgin Islands (U.S.),GDP per capita (constant 2015 US$),36309.743845,35850.743315,37313.048493,36983.431060,NaN,NaN,NaN


In [3]:
# Check data coverage per year
year_cols = ['YR2019', 'YR2020', 'YR2021', 'YR2022', 'YR2023', 'YR2024']

coverage = wb_raw[year_cols].notna().sum()
total = len(wb_raw)

print("Non-null values per year:")
for year, count in coverage.items():
    pct = (count / total) * 100
    print(f"  {year}: {count} ({pct:.0f}%)")


Non-null values per year:
  YR2019: 710 (89%)
  YR2020: 713 (89%)
  YR2021: 707 (89%)
  YR2022: 702 (88%)
  YR2023: 645 (81%)
  YR2024: 292 (37%)


In [4]:
# Year 2022 chosen: highest recent coverage (88%) + aligns with UNDP data year
wb_2022 = wb_raw[['economy', 'Country', 'series', 'YR2022']].copy()
wb_2022 = wb_2022.rename(columns={'YR2022': 'value', 'economy': 'country_code'})

print(f"Shape: {wb_2022.shape}")
wb_2022.head()

Shape: (798, 4)


,country_code,Country,series,value
0,ZWE,Zimbabwe,NY.GDP.PCAP.KD,1369.197615
1,ZMB,Zambia,NY.GDP.PCAP.KD,1298.848050
2,YEM,"Yemen, Rep.",NY.GDP.PCAP.KD,NaN
3,PSE,West Bank and Gaza,NY.GDP.PCAP.KD,3099.960901
4,VIR,Virgin Islands (U.S.),NY.GDP.PCAP.KD,36983.431060


In [5]:
# Pivot from long to wide format 
wb_wide = wb_2022.pivot_table(
    index=['country_code', 'Country'],
    columns='series',
    values='value'
).reset_index()

# Rename indicator codes to readable names
wb_wide = wb_wide.rename(columns={
    'NY.GDP.PCAP.KD':     'gdp_per_capita',
    'SE.XPD.TOTL.GD.ZS':  'education_expenditure_pct_gdp',
    'SH.XPD.CHEX.PC.CD':  'health_expenditure_per_capita'
})
# Standardize all column names to lowercase
wb_wide.columns = wb_wide.columns.str.lower()
print(f"Shape: {wb_wide.shape}")
wb_wide.head()

Shape: (261, 5)


series,country_code,country,gdp_per_capita,education_expenditure_pct_gdp,health_expenditure_per_capita
0,ABW,Aruba,29195.267023,NaN,NaN
1,AFE,Africa Eastern and Southern,1440.430366,3.697668,92.035288
2,AFG,Afghanistan,377.665627,NaN,80.651604
3,AFW,Africa Western and Central,1757.456590,2.891687,74.441253
4,AGO,Angola,2860.902519,2.385359,103.945061


In [6]:
# Remove regional aggregates (not individual countries)
regional_keywords = (
    'Africa Eastern|Africa Western|Arab World|Central Europe|'
    'East Asia|Europe & Central|European Union|high income|'
    'Latin America|Low income|Lower middle|Low & middle|'
    'Late-demographic|Early-demographic|Pre-demographic|'
    'Post-demographic|Middle East|Middle income|OECD members|'
    'South Asia|Sub-Saharan|IDA & IBRD|Upper middle|'
    'demographic dividend|^World$'
)
mask = wb_wide['country'].str.contains(regional_keywords, case=False)

print(f"Regional aggregates found: {mask.sum()}")
print(f"\nRemoved rows:")
print(wb_wide[mask]['country'].values)


print(f"\nCountries before filtering: {len(wb_wide)}")
wb_wide = wb_wide[~mask]
print(f"Countries after filtering: {len(wb_wide)}")

Regional aggregates found: 35

Removed rows:
['Africa Eastern and Southern' 'Africa Western and Central' 'Arab World'
 'Central Europe and the Baltics'
 'East Asia & Pacific (excluding high income)'
 'Early-demographic dividend' 'East Asia & Pacific'
 'Europe & Central Asia (excluding high income)' 'Europe & Central Asia'
 'European Union' 'High income' 'IDA & IBRD total'
 'Latin America & Caribbean (excluding high income)'
 'Latin America & Caribbean' 'Low income' 'Lower middle income'
 'Low & middle income' 'Late-demographic dividend'
 'Middle East, North Africa, Afghanistan & Pakistan' 'Middle income'
 'Middle East, North Africa, Afghanistan & Pakistan (excluding high income)'
 'OECD members' 'Pre-demographic dividend' 'Post-demographic dividend'
 'South Asia' 'Sub-Saharan Africa (excluding high income)'
 'Sub-Saharan Africa' 'East Asia & Pacific (IDA & IBRD countries)'
 'Europe & Central Asia (IDA & IBRD countries)'
 'Latin America & the Caribbean (IDA & IBRD countries)'
 'Middle E

In [7]:
wb_wide

series,country_code,country,gdp_per_capita,education_expenditure_pct_gdp,health_expenditure_per_capita
0,ABW,Aruba,29195.267023,NaN,NaN
2,AFG,Afghanistan,377.665627,NaN,80.651604
4,AGO,Angola,2860.902519,2.385359,103.945061
5,ALB,Albania,5867.650962,2.729770,506.869202
6,AND,Andorra,39780.415299,2.647280,3190.113281
...,...,...,...,...,...
256,XKX,Kosovo,4666.391378,NaN,NaN
257,YEM,"Yemen, Rep.",NaN,NaN,54.290363
258,ZAF,South Africa,5786.865053,6.155990,570.480286
259,ZMB,Zambia,1298.848050,3.658841,75.401283


In [8]:
# Map World Bank names to UNDP conventions 
# UNDP is the anchor dataset: all country names adapted to match

name_mapping = {
    'Bolivia':                    'Bolivia (Plurinational State of)',
    'Egypt, Arab Rep.':           'Egypt',
    'Iran, Islamic Rep.':         'Iran (Islamic Republic of)',
    'Korea, Rep.':                'Korea (Republic of)',
    'Congo, Dem. Rep.':           'Congo (Democratic Republic of the)',
    'Congo, Rep.':                'Congo',
    "Cote d'Ivoire":              "Côte d'Ivoire",
    'Turkiye':                    'Türkiye',
    'Tanzania':                   'Tanzania (United Republic of)',
    'Kyrgyz Republic':            'Kyrgyzstan',
    'Lao PDR':                    "Lao People's Democratic Republic",
    'Moldova':                    'Moldova (Republic of)',
    'Slovak Republic':            'Slovakia',
    'Eswatini':                   'Eswatini (Kingdom of)',
    'Venezuela, RB':              'Venezuela (Bolivarian Republic of)',
    'Yemen, Rep.':                'Yemen',
    'Gambia, The':                'Gambia',
    'Bahamas, The':               'Bahamas',
    'Micronesia, Fed. Sts.':      'Micronesia (Federated States of)',
    'Hong Kong SAR, China':       'Hong Kong, China (SAR)',
    'Macao SAR, China':           'China, Macao Special Administrative Region',
    'West Bank and Gaza':         'Palestine, State of',
    'Somalia, Fed. Rep.':         'Somalia',
    'St. Kitts and Nevis':        'Saint Kitts and Nevis',
    'St. Lucia':                  'Saint Lucia',
    'St. Vincent and the Grenadines': 'Saint Vincent and the Grenadines',
    'Puerto Rico (US)':           'Puerto Rico',
}
wb_wide = wb_wide.copy()
wb_wide['country'] = wb_wide['country'].replace(name_mapping)
print(f"Name mapping applied: {len(name_mapping)} country names standardized")

Name mapping applied: 27 country names standardized


In [9]:
# Save cleaned World Bank data
wb_wide.to_csv(PROCESSED_DATA / "wb_clean.csv", index=False)
print(f"World Bank cleaned data saved : {len(wb_wide)} countries")

World Bank cleaned data saved : 226 countries


## II Clean UNDP Data
Source: `undp_hdi_raw.csv`
Indicators: HDI, Mean Years of Schooling, Gender Inequality Index (GII)

In [10]:
# Load UNDP raw data 
undp_raw = pd.read_csv(RAW_DATA / "undp_hdi_raw.csv")

undp_raw.shape
undp_raw.head()

,Unnamed: 0,Unnamed: 1,Human Development Index (HDI),Unnamed: 3,Life expectancy at birth,Unnamed: 5,Expected years of schooling,Unnamed: 7,Mean years of schooling,Unnamed: 9,Gross national income (GNI) per capita,Unnamed: 11,GNI per capita rank minus HDI rank,Unnamed: 13,HDI rank
0,HDI rank,Country,Value,NaN,(years),NaN,(years),NaN,(years),NaN,(2017 PPP $),NaN,NaN,NaN,NaN
1,NaN,NaN,2022,NaN,2022,NaN,2022,a,2022,a,2022,NaN,2022,b,2021
2,NaN,VERY HIGH HUMAN DEVELOPMENT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Switzerland,0.967,NaN,84.255,NaN,16.5837307,NaN,13.90406622,c,69432.78669,NaN,6,NaN,1
4,2,Norway,0.966,NaN,83.393,NaN,18.63846016,d,13.06234279,c,69189.76165,NaN,6,NaN,2


In [11]:
undp_raw.columns

Index(['Unnamed: 0', 'Unnamed: 1', 'Human Development Index (HDI) ',
       'Unnamed: 3', 'Life expectancy at birth', 'Unnamed: 5',
       'Expected years of schooling', 'Unnamed: 7', 'Mean years of schooling',
       'Unnamed: 9', 'Gross national income (GNI) per capita', 'Unnamed: 11',
       'GNI per capita rank minus HDI rank', 'Unnamed: 13', 'HDI rank'],
      dtype='object')

In [12]:
# Select relevant columns and skip junk rows
undp_clean = undp_raw[['Unnamed: 1', 
                        'Human Development Index (HDI) ',  # note trailing space
                        'Mean years of schooling']].copy()

# Rename columns
undp_clean.columns = ['country', 'hdi', 'mean_years_schooling']

# Remove header rows and category labels (First 3 rows are headers/labels, not data)
undp_clean = undp_clean.iloc[3:].reset_index(drop=True)

undp_clean.head()

,country,hdi,mean_years_schooling
0,Switzerland,0.967,13.90406622
1,Norway,0.966,13.06234279
2,Iceland,0.959,13.76716995
3,"Hong Kong, China (SAR)",0.956,12.34776974
4,Denmark,0.952,12.96049023


In [13]:
# Check bottom of dataset 
undp_clean.tail()

,country,hdi,mean_years_schooling
264,Column 2: UNDESA (2022).,NaN,NaN
265,"Column 3: CEDLAS and The World Bank (2023), IC...",NaN,NaN
266,"Column 4: Barro and Lee (2018), ICF Macro Demo...",NaN,NaN
267,"Column 5: IMF (2023), UNDESA (2023), United Na...",NaN,NaN
268,Column 6: Calculated based on data in columns ...,NaN,NaN


In [14]:
# Convert numeric columns
cols_to_num = ['hdi', 'mean_years_schooling']
undp_clean[cols_to_num] = undp_clean[cols_to_num].apply(pd.to_numeric, errors='coerce')

# Remove footnotes - rows where both hdi and mean_years_schooling are NaN
undp_clean = undp_clean.dropna(subset=['hdi', 'mean_years_schooling'], how='all')

# Remove regional aggregates using keywords
regional_keywords = (
    'Sub-Saharan|Least developed|Small island|Organisation for Economic|'
    'World|Arab States|East Asia|Europe|Latin America|South Asia|'
    'Low|Medium|High|Very high|developing|OECD'
)

mask = undp_clean['country'].str.contains(regional_keywords, case=False, na=False)
print(f"Rows to remove: {mask.sum()}")
print(f"Removed: {undp_clean[mask]['country'].values}")

undp_clean = undp_clean[~mask].reset_index(drop=True)

undp_clean.tail()

Rows to remove: 15
Removed: ['Very high human development' 'High human development'
 'Medium human development' 'Low human development' 'Developing countries'
 'Arab States' 'East Asia and the Pacific' 'Europe and Central Asia'
 'Latin America and the Caribbean' 'South Asia' 'Sub-Saharan Africa'
 'Least developed countries' 'Small island developing states'
 'Organisation for Economic Co-operation and Development' 'World']


,country,hdi,mean_years_schooling
188,Chad,0.394,2.282753
189,Niger,0.394,1.341352
190,Central African Republic,0.387,3.953164
191,South Sudan,0.381,5.726140
192,Somalia,0.380,1.900300


In [15]:
# Save cleaned UNDP data 
undp_clean.to_csv(PROCESSED_DATA / "undp_clean.csv", index=False)
print(f"UNDP cleaned data saved: {len(undp_clean)} countries")

UNDP cleaned data saved: 193 countries


## III Clean WGI Data
Source: `wgi_government_effectiveness_raw.zip`
Indicator: Government Effectiveness Index

In [16]:
# Unzip and load WGI data 
import zipfile

# Open the zip and list its contents
with zipfile.ZipFile(RAW_DATA / "wgi_government_effectiveness_raw.zip", 'r') as z:
    print("Files inside zip:")
    print(z.namelist())

Files inside zip:
['Metadata_Indicator_API_GE.EST_DS2_en_csv_v2_4300.csv', 'API_GE.EST_DS2_en_csv_v2_4300.csv', 'Metadata_Country_API_GE.EST_DS2_en_csv_v2_4300.csv']


In [17]:
# Read WGI data file from zip 
with zipfile.ZipFile(RAW_DATA / "wgi_government_effectiveness_raw.zip", 'r') as z:
    with z.open('API_GE.EST_DS2_en_csv_v2_4300.csv') as f:
        wgi_raw = pd.read_csv(f, skiprows=4)
wgi_raw.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Government Effectiveness: Estimate,GE.EST,NaN,NaN,NaN,NaN,NaN,NaN,...,0.875970,1.013413,0.987694,1.074960,1.083876,1.105213,0.795441,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Government Effectiveness: Estimate,GE.EST,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Government Effectiveness: Estimate,GE.EST,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.386597,-1.501758,-1.519113,-1.611539,-1.670588,-1.880035,-1.987014,NaN,NaN,NaN
3,Africa Western and Central,AFW,Government Effectiveness: Estimate,GE.EST,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,Government Effectiveness: Estimate,GE.EST,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.951989,-1.005252,-1.128490,-1.257485,-1.126360,-1.026034,-1.008910,NaN,NaN,NaN


In [18]:
# Keep relevant columns only
wgi_clean = wgi_raw[['Country Name', 'Country Code', '2022']].copy()

# Rename columns 
wgi_clean.columns = ['country', 'country_code', 'government_effectiveness']

# Remove regional aggregates
regional_keywords = (
    'Africa Eastern|Africa Western|Arab World|Central Europe|'
    'East Asia|Europe & Central|European Union|high income|'
    'Latin America|Low income|Lower middle|Low & middle|'
    'Late-demographic|Early-demographic|Pre-demographic|'
    'Post-demographic|Middle East|Middle income|OECD|'
    'South Asia|Sub-Saharan|IDA & IBRD|Upper middle|'
    'demographic dividend|^World$'
)

mask = wgi_clean['country'].str.contains(regional_keywords, case=False, na=False)
wgi_clean = wgi_clean[~mask].reset_index(drop=True)

wgi_clean.head()

,country,country_code,government_effectiveness
0,Aruba,ABW,1.105213
1,Afghanistan,AFG,-1.880035
2,Angola,AGO,-1.026034
3,Albania,ALB,0.064541
4,Andorra,AND,1.495261


In [19]:
# Check for remaining non-country rows 
suspicious = wgi_clean[wgi_clean['government_effectiveness'].isna()]
print(f"Rows with missing government effectiveness: {len(suspicious)}")
print(suspicious['country'].values)

Rows with missing government effectiveness: 26
['Channel Islands' 'Caribbean small states' 'Curacao' 'Euro area'
 'Fragile and conflict affected situations' 'Faroe Islands' 'Gibraltar'
 'Heavily indebted poor countries (HIPC)' 'IBRD only' 'IDA total'
 'IDA blend' 'IDA only' 'Isle of Man' 'Not classified'
 'Least developed countries: UN classification' 'St. Martin (French part)'
 'Northern Mariana Islands' 'North America' 'New Caledonia'
 'Other small states' 'Pacific island small states' 'French Polynesia'
 'Small states' 'Sint Maarten (Dutch part)' 'Turks and Caicos Islands'
 'British Virgin Islands']


In [20]:
# Remove rows with missing government effectiveness
# All missing rows are either regional aggregates or territories

wgi_clean = wgi_clean.dropna(subset=['government_effectiveness']).reset_index(drop=True)

print(f"Shape after cleaning: {wgi_clean.shape}")
wgi_clean.head()

Shape after cleaning: (205, 3)


,country,country_code,government_effectiveness
0,Aruba,ABW,1.105213
1,Afghanistan,AFG,-1.880035
2,Angola,AGO,-1.026034
3,Albania,ALB,0.064541
4,Andorra,AND,1.495261


In [21]:
# Save cleaned WGI data 
wgi_clean.to_csv(PROCESSED_DATA / "wgi_clean.csv", index=False)
print(f"WGI cleaned data saved: {len(wgi_clean)} countries")

WGI cleaned data saved: 205 countries


## VI Clean Gender Inequality Index (GII)
Source: `undp_gii_raw.csv`
Indicator: Gender Inequality Index measures gender-based disadvantage

In [22]:
# Load GII raw data 
gii_raw = pd.read_csv(RAW_DATA / "undp_gii_raw.csv")

print(f"Shape: {gii_raw.shape}")
gii_raw

Shape: (261, 20)


,Unnamed: 0,Unnamed: 1,Value,Unnamed: 3,Rank,Unnamed: 5,"(deaths per 100,000 live births)",Unnamed: 7,"(births per 1,000 women ages 15–19)",Unnamed: 9,(% held by women),Unnamed: 11,(% ages 25 and older),Unnamed: 13,Unnamed: 14,Unnamed: 15,(% ages 15 and older),Unnamed: 17,Unnamed: 18,Unnamed: 19
0,HDI rank,Country,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Female,NaN,Male,NaN,Female,NaN,Male,NaN
1,NaN,NaN,2022,NaN,2022,NaN,2020,NaN,2022.000,NaN,2022,NaN,2022,b,2022,b,2022,NaN,2022,NaN
2,NaN,VERY HIGH HUMAN DEVELOPMENT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Switzerland,0.018,NaN,3,NaN,7.378754713,NaN,2.200,NaN,39.02439024,NaN,96.93972611,c,97.5174373,c,61.49,NaN,71.94,NaN
4,2,Norway,0.012,NaN,2,NaN,1.663741403,NaN,2.195,NaN,44.9704142,NaN,99.09403104,c,99.27497953,c,62.53,NaN,69.59,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256,NaN,"Column 3: WHO, UNICEF, UNFPA, World Bank Group...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
257,NaN,Column 4: UNDESA (2022).,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
258,NaN,Column 5: IPU (2023).,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
259,NaN,"Columns 6 and 7: Barro and Lee (2018), ICF Mac...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# Select relevant columns and skip junk rows 
gii_clean = gii_raw[['Unnamed: 1', 'Value']].copy()

# Rename columns
gii_clean.columns = ['country', 'gii']

# Remove header rows and category labels
gii_clean = gii_clean.iloc[3:].reset_index(drop=True)

# Convert to numeric 
gii_clean['gii'] = pd.to_numeric(gii_clean['gii'], errors='coerce')

# Remove footnotes and regional aggregates
gii_clean = gii_clean.dropna(subset=['gii']).reset_index(drop=True)

regional_keywords = (
    'Sub-Saharan|Least developed|Small island|Organisation for Economic|'
    'World|Arab States|East Asia|Europe|Latin America|South Asia|'
    'Low|Medium|High|Very high|developing|OECD'
)

mask = gii_clean['country'].str.contains(regional_keywords, case=False, na=False)
gii_clean = gii_clean[~mask].reset_index(drop=True)

print(f"Shape: {gii_clean.shape}")
gii_clean

Shape: (166, 2)


,country,gii
0,Switzerland,0.018
1,Norway,0.012
2,Iceland,0.039
3,Denmark,0.009
4,Sweden,0.023
...,...,...
161,Burundi,0.499
162,Mali,0.607
163,Chad,0.671
164,Niger,0.609


In [24]:
# Save cleaned GII data
gii_clean.to_csv(PROCESSED_DATA / "gii_clean.csv", index=False)
print(f"GII cleaned data saved: {len(gii_clean)} countries")

GII cleaned data saved: 166 countries


## V Merge All Datasets
Merging World Bank, UNDP, WGI and GII data into one master dataset.
Join key: country name



In [25]:
# Load cleaned datasets 
wb = pd.read_csv(PROCESSED_DATA / "wb_clean.csv")
undp = pd.read_csv(PROCESSED_DATA / "undp_clean.csv")
wgi = pd.read_csv(PROCESSED_DATA / "wgi_clean.csv")
gii = pd.read_csv(PROCESSED_DATA / "gii_clean.csv")

print(f"World Bank: {wb.shape}")
print(f"UNDP: {undp.shape}")
print(f"WGI: {wgi.shape}")
print(f"GII: {gii.shape}")

World Bank: (226, 5)
UNDP: (193, 3)
WGI: (205, 3)
GII: (166, 2)


In [26]:
#  Merge World Bank and WGI on country_code
master = pd.merge(wb, wgi[['country_code', 'government_effectiveness']], 
                  on ='country_code', 
                  how ='inner')

print(f"After WB + WGI merge: {master.shape}")
print(master.head())

After WB + WGI merge: (204, 6)
  country_code      country  gdp_per_capita  education_expenditure_pct_gdp  \
0          ABW        Aruba    29195.267023                            NaN   
1          AFG  Afghanistan      377.665627                            NaN   
2          AGO       Angola     2860.902519                       2.385359   
3          ALB      Albania     5867.650962                       2.729770   
4          AND      Andorra    39780.415299                       2.647280   

   health_expenditure_per_capita  government_effectiveness  
0                            NaN                  1.105213  
1                      80.651604                 -1.880035  
2                     103.945061                 -1.026034  
3                     506.869202                  0.064541  
4                    3190.113281                  1.495261  


In [27]:
# Merge with UNDP on country name
master = pd.merge(master, undp, 
                  on = 'country',
                  how = 'inner')
print(f"After adding UNDP: {master.shape}")
master.head()

After adding UNDP: (193, 8)


,country_code,country,gdp_per_capita,education_expenditure_pct_gdp,health_expenditure_per_capita,government_effectiveness,hdi,mean_years_schooling
0,AFG,Afghanistan,377.665627,NaN,80.651604,-1.880035,0.462,2.514790
1,AGO,Angola,2860.902519,2.385359,103.945061,-1.026034,0.591,5.844292
2,ALB,Albania,5867.650962,2.729770,506.869202,0.064541,0.789,10.121144
3,AND,Andorra,39780.415299,2.647280,3190.113281,1.495261,0.884,11.613440
4,ARE,United Arab Emirates,41828.555330,NaN,2240.905029,1.299128,0.937,12.773750


In [28]:
# Merge with GII on country name
master = pd.merge(master, gii, 
                  on='country', 
                  how='left')

print(f"After adding GII: {master.shape}")

After adding GII: (193, 9)


In [29]:
# Reorder columns logically 
master = master[['country_code', 'country', 'hdi', 
                 'gdp_per_capita', 'mean_years_schooling',
                 'education_expenditure_pct_gdp', 
                 'health_expenditure_per_capita',
                 'government_effectiveness', 
                 'gii']]

print(f"Final shape: {master.shape}")
master.head()

Final shape: (193, 9)


,country_code,country,hdi,gdp_per_capita,mean_years_schooling,education_expenditure_pct_gdp,health_expenditure_per_capita,government_effectiveness,gii
0,AFG,Afghanistan,0.462,377.665627,2.514790,NaN,80.651604,-1.880035,0.665
1,AGO,Angola,0.591,2860.902519,5.844292,2.385359,103.945061,-1.026034,0.520
2,ALB,Albania,0.789,5867.650962,10.121144,2.729770,506.869202,0.064541,0.116
3,AND,Andorra,0.884,39780.415299,11.613440,2.647280,3190.113281,1.495261,NaN
4,ARE,United Arab Emirates,0.937,41828.555330,12.773750,NaN,2240.905029,1.299128,0.035


## VI Missing Values Assessment
Reviewing missing data across all variables before deciding on a strategy.

In [30]:
# Missing values summary
missing = master.isnull().sum()
total = len(master)

print("Missing values per column:")
for col, count in missing.items():
    pct = (count / total) * 100
    print(f"  {col}: {count} ({pct:.1f}%)")

Missing values per column:
  country_code: 0 (0.0%)
  country: 0 (0.0%)
  hdi: 0 (0.0%)
  gdp_per_capita: 4 (2.1%)
  mean_years_schooling: 0 (0.0%)
  education_expenditure_pct_gdp: 39 (20.2%)
  health_expenditure_per_capita: 3 (1.6%)
  government_effectiveness: 0 (0.0%)
  gii: 27 (14.0%)


In [31]:
master

,country_code,country,hdi,gdp_per_capita,mean_years_schooling,education_expenditure_pct_gdp,health_expenditure_per_capita,government_effectiveness,gii
0,AFG,Afghanistan,0.462,377.665627,2.514790,NaN,80.651604,-1.880035,0.665
1,AGO,Angola,0.591,2860.902519,5.844292,2.385359,103.945061,-1.026034,0.520
2,ALB,Albania,0.789,5867.650962,10.121144,2.729770,506.869202,0.064541,0.116
3,AND,Andorra,0.884,39780.415299,11.613440,2.647280,3190.113281,1.495261,NaN
4,ARE,United Arab Emirates,0.937,41828.555330,12.773750,NaN,2240.905029,1.299128,0.035
...,...,...,...,...,...,...,...,...,...
188,WSM,Samoa,0.702,4095.158683,11.367356,6.159788,259.954041,0.307864,0.406
189,YEM,Yemen,0.424,NaN,2.776826,NaN,54.290363,-2.233465,0.820
190,ZAF,South Africa,0.717,5786.865053,11.606970,6.155990,570.480286,-0.109028,0.401
191,ZMB,Zambia,0.569,1298.848050,7.284893,3.658841,75.401283,-0.638017,0.526


## VI Missing Values Strategy

| Variable | Missing | Action | Reason |
|---|---|---|---|
| `gdp_per_capita` | 4 (2.1%) | Drop rows | Core proxy variable to GNI , unusable without it |
| `health_expenditure_per_capita` | 3 (1.6%) | Drop rows | Core variable , negligible loss |
| `education_expenditure_pct_gdp` | 39 (20.2%) | Keep NaN | Too many to drop, available cases used in regression |
| `gii` | 27 (14.0%) | Keep NaN | Supplementary variable, expected from left join |

**Total rows dropped: 7 out of 193 (≈3.6%) within acceptable limits for listwise deletion.**

In [32]:
# Drop rows missing core variables 
# GDP and health are essential for regression - drop if missing
core_vars = ['gdp_per_capita', 'health_expenditure_per_capita']

before = len(master)
master = master.dropna(subset=core_vars).reset_index(drop=True)
after = len(master)

print(f"Rows before: {before}")
print(f"Rows after:  {after}")
print(f"Rows dropped: {before - after}")

Rows before: 193
Rows after:  186
Rows dropped: 7


In [33]:
# Final missing values check 
missing = master.isnull().sum()
total = len(master)

print("Final missing values:")
for col, count in missing.items():
    pct = (count / total) * 100
    print(f"  {col}: {count} ({pct:.1f}%)")

Final missing values:
  country_code: 0 (0.0%)
  country: 0 (0.0%)
  hdi: 0 (0.0%)
  gdp_per_capita: 0 (0.0%)
  mean_years_schooling: 0 (0.0%)
  education_expenditure_pct_gdp: 34 (18.3%)
  health_expenditure_per_capita: 0 (0.0%)
  government_effectiveness: 0 (0.0%)
  gii: 23 (12.4%)


## VII Export Master Dataset
Final dataset: 186 countries, 9 variables, reference year 2022.

In [34]:
# Export master dataset 
master.to_csv(PROCESSED_DATA / "master_dataset.csv", index=False)
print(f"Master dataset exported: {master.shape[0]} countries, {master.shape[1]} variables")

Master dataset exported: 186 countries, 9 variables


In [35]:
import wbgapi as wb

# Fetch country metadata including region and income level 
countries = wb.economy.DataFrame()
print(countries.columns.tolist())
countries.head()

['name', 'aggregate', 'longitude', 'latitude', 'region', 'adminregion', 'lendingType', 'incomeLevel', 'capitalCity']


,name,aggregate,longitude,latitude,region,adminregion,lendingType,incomeLevel,capitalCity
id,,,,,,,,,
ABW,Aruba,False,-70.0167,12.51670,LCN,,LNX,HIC,Oranjestad
AFE,Africa Eastern and Southern,True,NaN,NaN,,,,,
AFG,Afghanistan,False,69.1761,34.52280,MEA,MNA,IDX,LIC,Kabul
AFW,Africa Western and Central,True,NaN,NaN,,,,,
AGO,Angola,False,13.2420,-8.81155,SSF,SSA,IBD,LMC,Luanda


In [36]:
# Extract region and income metadata 
region_meta = countries[countries['aggregate'] == False][['name', 'region', 'incomeLevel']].reset_index()
region_meta.columns = ['country_code', 'country_name', 'wb_region', 'wb_income_level']

# Map World Bank region codes to full names 
region_map = {
    'EAS': 'East Asia & Pacific',
    'ECS': 'Europe & Central Asia',
    'LCN': 'Latin America & Caribbean',
    'MEA': 'Middle East & North Africa',
    'NAC': 'North America',
    'SAS': 'South Asia',
    'SSF': 'Sub-Saharan Africa'
}

income_map = {
    'HIC': 'High income',
    'UMC': 'Upper middle income',
    'LMC': 'Lower middle income',
    'LIC': 'Low income'
}

region_meta['region'] = region_meta['wb_region'].map(region_map)
region_meta['income_level'] = region_meta['wb_income_level'].map(income_map)

print(region_meta[['country_code', 'region', 'income_level']].head(10))
print(f"\nShape: {region_meta.shape}")

  country_code                      region         income_level
0          ABW   Latin America & Caribbean          High income
1          AFG  Middle East & North Africa           Low income
2          AGO          Sub-Saharan Africa  Lower middle income
3          ALB       Europe & Central Asia  Upper middle income
4          AND       Europe & Central Asia          High income
5          ARE  Middle East & North Africa          High income
6          ARG   Latin America & Caribbean  Upper middle income
7          ARM       Europe & Central Asia  Upper middle income
8          ASM         East Asia & Pacific          High income
9          ATG   Latin America & Caribbean          High income

Shape: (217, 6)


## Region & Income Level Enrichment
A lightweight post‑processing step adds World Bank regional classifications to the master dataset for use in the BI dashboard.

In [37]:
# Region & Income Level Enrichment 
meta = pd.read_csv(RAW_DATA / "wb_country_metadata_raw.csv")[['country_code', 'region', 'income_level']]

master = pd.read_csv(PROCESSED_DATA / "master_dataset.csv")

# Drop existing enrichment columns if already present
master = master.drop(columns=['region', 'income_level'], errors='ignore')

master = pd.merge(master, meta, on='country_code', how='left')

print(f"Missing region:       {master['region'].isna().sum()}")
print(f"Missing income_level: {master['income_level'].isna().sum()}")
print(f"Shape: {master.shape}")

master.to_csv(PROCESSED_DATA / "master_dataset.csv", index=False)
print("Saved")

Missing region:       0
Missing income_level: 2
Shape: (186, 11)
Saved


In [38]:
# Check which countries with missing income level (will be handled in Power BI with DAX as unclassified income level)
print(master[master['income_level'].isna()][['country_code', 'country']])

    country_code                             country
53           ETH                            Ethiopia
179          VEN  Venezuela (Bolivarian Republic of)
